# Preprocessing Pipeline
### Chest X-Ray Classification: Normal vs Pneumonia vs COVID-19
This notebook builds the preprocessing pipeline: resizing images to 224x224, converting all images to RGB, and normalising pixel values.

In [17]:
#importing the libraries
import os 
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image #to open images
from pathlib import Path #to work with paths
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch

In [18]:
data_dir = Path('../data/Dataset')
train_dir = data_dir/ 'Train_Validation'
test_dir = data_dir / 'Test'
classes = ['COVID', 'Normal', 'Pneumonia']

In [19]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [20]:
# Test transform on one image
sample_path = list((train_dir / 'COVID').glob('*'))[0]
img = Image.open(sample_path).convert('RGB')
tensor = transform(img)
print("Original image size:", img.size)
print("Tensor shape:", tensor.shape)
print("Tensor min:", tensor.min().item())
print("Tensor max:", tensor.max().item())

Original image size: (256, 256)
Tensor shape: torch.Size([3, 224, 224])
Tensor min: -1.621286153793335
Tensor max: 2.6225709915161133


In [21]:
def preprocess_image(path): 
    img = Image.open(path).convert('RGB') 
    tensor = transform(img)
    return tensor

In [22]:
sample_path = list((train_dir / 'COVID').glob('*'))[0]
tensor = preprocess_image(sample_path)
print(tensor.shape)

torch.Size([3, 224, 224])


In [26]:
class XRayDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for label, cls in enumerate(classes):
            class_dir = self.data_dir / cls
            for img_path in class_dir.glob('*'):
                self.image_paths.append(img_path)
                self.labels.append(label)

                
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img_path = self.image_paths[index]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        label = self.labels[index]
        return image, label

In [27]:
train_dataset = XRayDataset(train_dir, transform=transform)
print("Total training images:", len(train_dataset))

Total training images: 2886


In [28]:
image, label = train_dataset[0]
print("Image shape:", image.shape)
print("Label:", label)
print("Class:", classes[label])

Image shape: torch.Size([3, 224, 224])
Label: 0
Class: COVID


In [29]:
test_dataset = XRayDataset(test_dir, transform=transform)
print("Total test images:", len(test_dataset))

Total test images: 341


In [30]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Number of training batches:", len(train_loader))
print("Number of test batches:", len(test_loader))

Number of training batches: 91
Number of test batches: 11


In [31]:
images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch labels:", labels)

Batch image shape: torch.Size([32, 3, 224, 224])
Batch labels: tensor([2, 1, 1, 0, 2, 1, 1, 2, 0, 1, 2, 0, 1, 2, 2, 1, 0, 0, 1, 0, 2, 2, 2, 1,
        0, 0, 0, 2, 2, 2, 2, 0])


Transform pipeline — resize, convert to tensor, normalise
Custom Dataset class — stores paths and labels, preprocesses on demand
DataLoaders — batches, shuffles, and serves data to the model